In [17]:
import jax.numpy as jnp
import jax as j 
import matplotlib.pyplot as plt
import numpy as np


A = 1 
alpha = 0.1 
gamma = 0.5
epsilon = 0.01
iapp = 0.5 
v0 = 0.0
w0 = 0.5

tf = 5 #tempo final 
h = [0.01,0.001] #passos de tempo 


In [ ]:
def f(v):
    return A* v *(alpha - v) * (v - 1)

def sistemaDeEquacoes(estado):
    v, w = estado
    dvdt = 1/epsilon * (f(v) - w + iapp)
    dwdt = v - gamma*w 
    return jnp.array([dvdt, dwdt])

# Jacobiano pré-compilado com JIT — calculado uma única vez
jacobiano_sistema = j.jit(j.jacfwd(sistemaDeEquacoes))

def newton(sistema, y_anterior, h, x0, maxit=10, tol=1e-6):
    x = x0
    
    for i in range(maxit):
        # F(y) = y - y_anterior - h * sistema(y)
        fx = x - y_anterior - h * sistema(x)
        
        # J_F = I - h * J_sistema
        J_sistema = jacobiano_sistema(x)
        J = jnp.eye(len(x)) - h * J_sistema
        
        delta = jnp.linalg.solve(J, -fx)
        x = x + delta
       
        if jnp.linalg.norm(delta) < tol:
            break
    
    return x


def eulerImplicito(sistema, h, y0, t):
    n = len(t)
    sol = np.zeros((n, len(y0)))
    sol[0] = np.array(y0)
    
    for i in range(n-1):
        x0 = jnp.array(sol[i])
        y_anterior = x0
        resultado = newton(sistema, y_anterior, h, x0)
        sol[i+1] = np.array(resultado)
    
    return sol

In [ ]:
for i in h:
    t = np.arange(0, tf + i, i)
    y0_inicial = jnp.array([v0, w0])
    
    solucao = eulerImplicito(sistemaDeEquacoes, i, y0_inicial, t)
    
    plt.figure(figsize=(10, 5))
    plt.plot(t, solucao[:,0], label='Potencial (v)', color='blue')
    plt.plot(t, solucao[:,1], label='Recuperação (w)', color='red', linestyle='--')
    plt.title(f"Euler Implícito (Newton via JAX) | h = {i}")
    plt.xlabel('Tempo')
    plt.grid(True)
    plt.legend()
    plt.show()